# Explainable Machine Learning for Diabetes Status Assessment

Classification, regression, patient subgroup analysis and Random Forest SHAP interpretation.

**Dataset:** the 390-record Kaggle diagnostic-measures export, empirically matched to the primary 403-record Virginia cohort. This is not the Pima dataset. The target is recorded status, not future incidence.


## Abstract

This study assesses recorded diabetes status using 14 diagnostic and demographic predictors. Training CV selects Random Forest; its training-only OOF threshold of 0.45 yields held-out sensitivity 0.50 (95% test bootstrap CI 0.25-0.75). Excluding Glucose reduces the selected forest's test AUC from 0.9337 to 0.6919. Regression, clustering and verified Random Forest SHAP explanations provide additional descriptive evidence. The small test cohort and upstream preprocessing limit screening claims.


## Introduction

Diabetes screening is a clinically important task because delayed detection increases the risk of cardiovascular disease, renal complications, neuropathy, and preventable hospitalizations. Machine learning can support earlier identification by learning nonlinear relationships among routine clinical measures, but high performance alone is not enough in healthcare. Screening models must also be interpretable, reasonably calibrated, and deployable within workflows that prioritize patient safety.

This study intentionally combines supervised and unsupervised learning. Classification addresses patient-level diabetes prediction, regression explores whether continuous metabolic information can be estimated from correlated variables, and clustering investigates whether meaningful patient phenotypes emerge without outcome labels. Explainable AI is added because clinicians, patients, and governance teams need to understand which features drive elevated or reduced model-estimated risk.


## Research Questions

1. Which clinical features contribute most strongly to diabetes prediction in this dataset?
2. Which classification model provides the strongest balance between discrimination, recall, and calibration?
3. Does threshold tuning materially improve screening-oriented recall while preserving acceptable precision?
4. Can nonlinear regression models estimate glucose more effectively than a linear baseline?
5. Do unsupervised clustering methods reveal clinically meaningful patient subgroups with different diabetes prevalence profiles?
6. How do global and local explainability analyses support practical healthcare interpretation of model behavior?


## Reproducibility Setup

The project uses a fixed random seed, modular helper functions under `src/`, and automatic figure and table export under `outputs/`. The notebook assumes the CSV file is available at `data/raw/diabetes.csv`.


In [1]:
from pathlib import Path
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.base import clone
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering import run_clustering_analysis
from src.config import (
    EXPECTED_COLUMNS,
    FEATURE_COLUMNS,
    FIGURES_DIR,
    MODELS_DIR,
    RANDOM_STATE,
    RAW_DATA_PATH,
    TABLES_DIR,
    TEST_SIZE,
    ensure_project_directories,
)
from src.data import get_regression_frame, load_diabetes_dataset, prepare_dataset
from src.explainability import build_explainability_outputs
from src.models import run_classification_experiment, run_regression_experiment
from src.plots import (
    plot_boxplots,
    plot_calibration_curves,
    plot_class_distribution,
    plot_clustering_diagnostics,
    plot_confusion_matrices,
    plot_correlation_heatmap,
    plot_decision_boundary,
    plot_feature_importance,
    plot_histograms,
    plot_pairplot,
    plot_pca_clusters,
    plot_regression_diagnostics,
    plot_roc_curves,
    plot_scatter_panel,
    plot_threshold_sweep,
    plot_zero_audit,
    save_figure,
    set_publication_style,
)
from src.preprocess import build_classification_pipeline, split_classification_data

ensure_project_directories()
set_publication_style()

print(f"Project root: {PROJECT_ROOT}")
print(f"Expected dataset path: {RAW_DATA_PATH}")
print(f"Random seed: {RANDOM_STATE}")


Project root: C:\Users\mhral\Documents\New project\diabetes-mini-study
Expected dataset path: C:\Users\mhral\Documents\New project\diabetes-mini-study\data\raw\diabetes.csv
Random seed: 42


In [2]:
version_rows = []
for package_name, module in [
    ("pandas", pd),
    ("numpy", np),
    ("seaborn", sns),
]:
    version_rows.append({"Package": package_name, "Version": getattr(module, "__version__", "n/a")})

for optional_package in ("sklearn", "xgboost", "shap"):
    try:
        module = __import__(optional_package)
        version_rows.append({"Package": optional_package, "Version": getattr(module, "__version__", "available")})
    except ImportError:
        version_rows.append({"Package": optional_package, "Version": "not installed"})

versions = pd.DataFrame(version_rows)
display(versions)
versions.to_csv(TABLES_DIR / "package_versions.csv", index=False)


,Package,Version
0,pandas,3.0.2
1,numpy,2.4.4
2,seaborn,0.13.2
3,sklearn,1.8.0
4,xgboost,3.2.0
5,shap,0.51.0


## Dataset Description

The local export has 390 raw and analytic records, zero duplicates removed, 14 predictors and Outcome. All 390 labels agree with glycosylated hemoglobin >7.0 in uniquely compatible primary records. Glucose is a distinct source variable; glycosylated hemoglobin is not a predictor. Publisher adjudication and timing remain undocumented. Thirteen matching primary patients have missing predictors that are filled in the local export. See the provenance audit for exact matching rules. Test counts are 12 positive and 66 negative.


In [3]:
feature_dictionary = pd.DataFrame([{"Feature":"Cholesterol","Meaning":"Total cholesterol (laboratory units not verified for the export)."},{"Feature":"Glucose","Meaning":"Stabilized glucose in the matched primary data; assay timing not established."},{"Feature":"HDLChol","Meaning":"High-density lipoprotein cholesterol."},{"Feature":"CholHDLRatio","Meaning":"Total cholesterol divided by HDL cholesterol."},{"Feature":"Age","Meaning":"Patient age in years."},{"Feature":"Gender","Meaning":"Recorded gender category, one-hot encoded."},{"Feature":"Height","Meaning":"Height, recorded in inches."},{"Feature":"Weight","Meaning":"Weight, recorded in pounds."},{"Feature":"BMI","Meaning":"Body mass index derived from height and weight."},{"Feature":"SystolicBP","Meaning":"Systolic blood pressure in mmHg."},{"Feature":"DiastolicBP","Meaning":"Diastolic blood pressure in mmHg."},{"Feature":"Waist","Meaning":"Waist circumference in inches."},{"Feature":"Hip","Meaning":"Hip circumference in inches."},{"Feature":"WaistHipRatio","Meaning":"Waist divided by hip circumference."},{"Feature":"Outcome","Meaning":"Publisher diabetes status; all 390 labels agree with matched primary glycosylated hemoglobin >7.0."}])
display(feature_dictionary)
feature_dictionary.to_csv(TABLES_DIR / "feature_dictionary.csv", index=False)


,Feature,Meaning
0,Pregnancies,Number of pregnancies.
1,Glucose,Plasma glucose concentration after an oral glu...
2,BloodPressure,Diastolic blood pressure in mm Hg.
3,SkinThickness,Triceps skin fold thickness in millimeters.
4,Insulin,2-hour serum insulin concentration.
5,BMI,"Body mass index, a weight-for-height adiposity..."
6,DiabetesPedigreeFunction,Composite proxy of family history related diab...
7,Age,Patient age in years.
8,Outcome,Binary diabetes status label used for classifi...


In [2]:
raw_df = load_diabetes_dataset()
dataset_bundle = prepare_dataset(raw_df)
clean_df = dataset_bundle.clean.copy()

metadata_table = pd.DataFrame([dataset_bundle.metadata])
display(raw_df.head())
display(metadata_table)
display(dataset_bundle.data_quality)

metadata_table.to_csv(TABLES_DIR / "dataset_metadata.csv", index=False)
dataset_bundle.data_quality.to_csv(TABLES_DIR / "data_quality_summary.csv", index=False)
dataset_bundle.dataset_summary.to_csv(TABLES_DIR / "dataset_summary.csv")
dataset_bundle.zero_audit.to_csv(TABLES_DIR / "zero_as_missing_audit.csv", index=False)
dataset_bundle.outlier_summary.to_csv(TABLES_DIR / "outlier_summary.csv", index=False)
dataset_bundle.class_balance.to_csv(TABLES_DIR / "class_balance.csv", index=False)


,Cholesterol,Glucose,HDLChol,CholHDLRatio,Age,Gender,Height,Weight,BMI,SystolicBP,DiastolicBP,Waist,Hip,WaistHipRatio,Outcome
0,193,77,49,3.9,19,Female,61,119,22.5,118,70,32,38,0.84,0
1,146,79,41,3.6,19,Female,60,135,26.4,108,58,33,40,0.83,0
2,217,75,54,4.0,20,Female,67,187,29.3,110,72,40,45,0.89,0
3,226,97,70,3.2,20,Female,64,114,19.6,122,64,31,39,0.79,0
4,164,91,67,2.4,20,Female,70,141,20.2,122,86,32,39,0.82,0


,rows_raw,rows_clean,columns,duplicates_removed
0,390,390,15,0


,Feature,Dtype,RawMissing,ZeroAsMissingCandidates,CleanMissing,MissingPercentAfterCleaning
0,Cholesterol,int64,0,0,0,0.0
1,Glucose,int64,0,0,0,0.0
2,HDLChol,int64,0,0,0,0.0
3,CholHDLRatio,float64,0,0,0,0.0
4,Age,int64,0,0,0,0.0
5,Gender,str,0,0,0,0.0
6,Height,int64,0,0,0,0.0
7,Weight,int64,0,0,0,0.0
8,BMI,float64,0,0,0,0.0
9,SystolicBP,int64,0,0,0,0.0


## Data Preprocessing

No local missing values or suspicious zeros required imputation. Numeric median and categorical mode imputers are configured safeguards, with Gender one-hot encoding. RobustScaler is used for linear/kernel models only. Transformations are fitted within training folds. Upstream filling of 13 patients' primary measurements is separate and cannot be reconstructed. IQR outliers are retained. The stratified split uses 312 training and 78 test patients, seed 42.


In [3]:
X_train, X_test, y_train, y_test = split_classification_data(clean_df)
correlation_matrix = clean_df[EXPECTED_COLUMNS].corr(numeric_only=True)

print(f"Training shape: {X_train.shape}, Test shape: {X_test.shape}")
display(dataset_bundle.class_balance)
display(dataset_bundle.outlier_summary)
display(correlation_matrix.round(3))

correlation_matrix.to_csv(TABLES_DIR / "correlation_matrix.csv")


Training shape: (312, 14), Test shape: (78, 14)


,Outcome,Label,Count,Percent
0,0,No Diabetes,330,84.62
1,1,Diabetes,60,15.38


,Feature,Q1,Q3,IQR,LowerFence,UpperFence,OutlierCount,OutlierPercent
0,Glucose,81.00,107.750,26.750,40.875,147.875,50,12.82
1,HDLChol,38.00,59.000,21.000,6.500,90.500,13,3.33
2,DiastolicBP,75.00,90.000,15.000,52.500,112.500,13,3.33
3,SystolicBP,122.00,148.000,26.000,83.000,187.000,12,3.08
4,Hip,39.00,46.000,7.000,28.500,56.500,12,3.08
5,Cholesterol,179.00,229.000,50.000,104.000,304.000,11,2.82
6,Weight,150.25,200.000,49.750,75.625,274.625,10,2.56
7,BMI,24.10,32.275,8.175,11.838,44.537,8,2.05
8,CholHDLRatio,3.20,5.400,2.200,-0.100,8.700,6,1.54
9,WaistHipRatio,0.83,0.930,0.100,0.680,1.080,4,1.03


,Cholesterol,Glucose,HDLChol,CholHDLRatio,Age,Height,Weight,BMI,SystolicBP,DiastolicBP,Waist,Hip,WaistHipRatio,Outcome
Cholesterol,1.000,0.158,0.193,0.476,0.247,-0.064,0.062,0.092,0.208,0.166,0.134,0.093,0.092,0.204
Glucose,0.158,1.000,-0.158,0.282,0.294,0.098,0.190,0.129,0.163,0.020,0.222,0.138,0.185,0.689
HDLChol,0.193,-0.158,1.000,-0.682,0.028,-0.087,-0.292,-0.242,0.032,0.078,-0.277,-0.224,-0.159,-0.123
CholHDLRatio,0.476,0.282,-0.682,1.000,0.163,0.081,0.279,0.228,0.116,0.038,0.313,0.209,0.243,0.273
Age,0.247,0.294,0.028,0.163,1.000,-0.082,-0.057,-0.009,0.453,0.069,0.151,0.005,0.275,0.302
Height,-0.064,0.098,-0.087,0.081,-0.082,1.000,0.255,-0.260,-0.041,0.044,0.057,-0.096,0.253,0.023
Weight,0.062,0.190,-0.292,0.279,-0.057,0.255,1.000,0.860,0.097,0.166,0.848,0.827,0.250,0.163
BMI,0.092,0.129,-0.242,0.228,-0.009,-0.260,0.860,1.000,0.121,0.145,0.811,0.882,0.101,0.145
SystolicBP,0.208,0.163,0.032,0.116,0.453,-0.041,0.097,0.121,1.000,0.604,0.211,0.155,0.138,0.199
DiastolicBP,0.166,0.020,0.078,0.038,0.069,0.044,0.166,0.145,0.604,1.000,0.166,0.144,0.078,0.049


## Exploratory Data Analysis

The EDA below focuses on class balance, clinically implausible zeros, marginal feature distributions, outlier structure, and pairwise relationships that are often clinically informative. Particular attention is paid to glucose and BMI because they are central to metabolic risk, while age and insulin help reveal heterogeneity in disease burden and endocrine dysfunction.


In [6]:
zero_fig = plot_zero_audit(dataset_bundle.zero_audit)
save_figure(zero_fig, "zero_audit.png")
plt.show()

class_fig = plot_class_distribution(clean_df)
save_figure(class_fig, "class_distribution.png")
plt.show()

hist_fig = plot_histograms(clean_df, FEATURE_COLUMNS)
save_figure(hist_fig, "histograms_by_outcome.png")
plt.show()

box_fig = plot_boxplots(clean_df, FEATURE_COLUMNS)
save_figure(box_fig, "boxplots_by_outcome.png")
plt.show()

corr_fig = plot_correlation_heatmap(correlation_matrix)
save_figure(corr_fig, "correlation_heatmap.png")
plt.show()

scatter_fig = plot_scatter_panel(clean_df)
save_figure(scatter_fig, "focused_scatter_panel.png")
plt.show()

pairplot_columns = [column for column in ["Glucose", "BMI", "Age", "WaistHipRatio"] if column in clean_df.columns]
pairplot_grid = plot_pairplot(clean_df.dropna(subset=pairplot_columns))
pairplot_grid.figure.savefig(FIGURES_DIR / "pairplot_key_features.png", bbox_inches="tight", dpi=300)
plt.show()


**Interpretation guide for the EDA:**

- The zero-audit figure quantifies how often impossible zero values appear, highlighting why naive missing-value checks understate data quality issues.
- Histograms and boxplots help assess skew, overlap between diabetes classes, and the extent to which high-risk groups concentrate at elevated glucose, BMI, insulin, or age values.
- The correlation heatmap identifies dependencies relevant to both predictive modeling and clinical interpretation.
- The focused scatter panel helps visualize whether glucose and BMI jointly separate risk groups and whether age-insulin relationships suggest subgroup heterogeneity.


## Classification Task

Four model families are tuned with training-only repeated 5-fold, 3-repeat CV ROC-AUC. These are tuning scores, not nested performance estimates. Screening thresholds use a separate nested 5-fold training OOF loop and are fixed before test evaluation. Default estimator decisions differ from the training-derived threshold. The revision adds average precision, sensitivity, specificity, PPV, NPV, and 2,000-replicate fixed-test bootstrap intervals, plus a Glucose ablation using the same split and grids.


In [7]:
classification_results = run_classification_experiment(X_train, X_test, y_train, y_test)
classification_table = classification_results["baseline_table"]
best_run = classification_results["best_run"]
best_tree_run = classification_results["best_tree_run"]
best_smote_run = classification_results["best_smote_run"]
threshold_table = classification_results["threshold_table"]
recommended_threshold = classification_results["recommended_threshold"]
threshold_test = classification_results["threshold_test_evaluation"]
threshold_test_row = {"Threshold": threshold_test["Threshold"], "Precision": threshold_test["Precision"], "Recall": threshold_test["Recall"], "F1Score": threshold_test["F1Score"], "Specificity": threshold_test["Counts"]["tn"] / (threshold_test["Counts"]["tn"] + threshold_test["Counts"]["fp"]), "FalseNegatives": threshold_test["Counts"]["fn"], "FalsePositives": threshold_test["Counts"]["fp"]}
paired_test = classification_results["paired_test"]

display(classification_table)
display(threshold_table)
display(pd.DataFrame([best_smote_run]).drop(columns=["BestEstimator", "Spec", "BestParams", "ConfusionMatrix", "Probabilities", "Predictions", "RocCurve", "CalibrationCurve", "Counts"], errors="ignore"))

classification_table.to_csv(TABLES_DIR / "classification_model_comparison.csv", index=False)
threshold_table.to_csv(TABLES_DIR / "classification_threshold_sweep.csv", index=False)
pd.DataFrame([recommended_threshold]).to_csv(TABLES_DIR / "training_oof_selected_threshold.csv", index=False)
pd.DataFrame([threshold_test_row]).to_csv(TABLES_DIR / "recommended_threshold.csv", index=False)
pd.DataFrame({"PatientIndex": y_train.index, "Outcome": y_train.to_numpy(), "OOFProbability": classification_results["oof_probabilities"]}).to_csv(TABLES_DIR / "training_oof_predictions.csv", index=False)
pd.DataFrame({"PatientIndex": y_test.index, "Outcome": y_test.to_numpy(), "Probability": threshold_test["Probabilities"], "Prediction": threshold_test["Predictions"]}).to_csv(TABLES_DIR / "threshold_test_predictions.csv", index=False)
display(pd.DataFrame([threshold_test_row]))
pd.DataFrame([{"Model": best_smote_run["Model"], "Precision": best_smote_run["Precision"], "Recall": best_smote_run["Recall"], "BrierScore": best_smote_run["BrierScore"]}]).to_csv(TABLES_DIR / "smote_sensitivity.csv", index=False)

paired_test_table = pd.DataFrame([
    {
        key: value
        for key, value in paired_test.items()
        if key not in {"scores_a", "scores_b"}
    }
])
paired_test_table = paired_test_table.rename(columns={"model_a": "Model A", "model_b": "Model B", "median_a": "Median ROC-AUC (A)", "median_b": "Median ROC-AUC (B)", "wilcoxon_statistic": "Wilcoxon Statistic", "p_value": "p-Value"})
paired_test_table.to_csv(TABLES_DIR / "paired_model_comparison.csv", index=False)

joblib.dump(best_run["BestEstimator"], MODELS_DIR / "best_classifier.joblib")

display(
    Markdown(
        f"**Best classification model:** {best_run['Model']} with test ROC-AUC **{best_run['ROCAUC']:.3f}**, "
        f"recall **{best_run['Recall']:.3f}**, and F1-score **{best_run['F1Score']:.3f}**. "
        f"This model produced **{best_run['Counts']['fn']} false negatives** and **{best_run['Counts']['fp']} false positives** on the held-out set."
    )
)

if "p_value" in paired_test:
    display(
        Markdown(
            f"The paired Wilcoxon comparison between **{paired_test['model_a']}** and **{paired_test['model_b']}** "
            f"yielded a p-value of **{paired_test['p_value']:.4f}**, which should be interpreted as supportive rather than definitive evidence because the dataset is small."
        )
    )


,Model,CV ROC-AUC,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,Brier Score,False Negatives,False Positives
0,Random Forest,0.9327,0.8974,0.7500,0.5000,0.6000,0.9337,0.0819,6,2
1,Support Vector Machine,0.9236,0.9103,0.6667,0.8333,0.7407,0.9747,0.0496,2,5
2,Logistic Regression,0.9193,0.9103,0.7778,0.5833,0.6667,0.9798,0.0522,5,2
3,XGBoost,0.9148,0.8974,0.7500,0.5000,0.6000,0.9394,0.0738,6,2


,Threshold,Precision,Recall,F1Score,Specificity,FalseNegatives,FalsePositives
0,0.10,0.247312,0.958333,0.393162,0.469697,2,140
1,0.15,0.310811,0.958333,0.469388,0.613636,2,102
2,0.20,0.400000,0.958333,0.564417,0.738636,2,69
3,0.25,0.474227,0.958333,0.634483,0.806818,2,51
4,0.30,0.537500,0.895833,0.671875,0.859848,5,37
5,0.35,0.585714,0.854167,0.694915,0.890152,7,29
6,0.40,0.672131,0.854167,0.752294,0.924242,7,20
7,0.45,0.735849,0.812500,0.772277,0.946970,9,14
8,0.50,0.750000,0.750000,0.750000,0.954545,12,12
9,0.55,0.789474,0.625000,0.697674,0.969697,18,8


,Model,CVBestScore,Accuracy,Precision,Recall,F1Score,ROCAUC,BrierScore
0,Random Forest + SMOTE,0.931187,0.897436,0.666667,0.666667,0.666667,0.931818,0.103607


,Threshold,Precision,Recall,F1Score,Specificity,FalseNegatives,FalsePositives
0,0.45,0.666667,0.5,0.571429,0.954545,6,3


**Best classification model:** Random Forest with test ROC-AUC **0.934**, recall **0.500**, and F1-score **0.600**. This model produced **6 false negatives** and **2 false positives** on the held-out set.

The paired Wilcoxon comparison between **Random Forest** and **Support Vector Machine** yielded a p-value of **0.3591**, which should be interpreted as supportive rather than definitive evidence because the dataset is small.

In [8]:
roc_fig = plot_roc_curves(classification_results["runs"])
save_figure(roc_fig, "classification_roc_comparison.png")
plt.show()

calibration_fig = plot_calibration_curves(classification_results["runs"])
save_figure(calibration_fig, "classification_calibration_curves.png")
plt.show()

confusion_fig = plot_confusion_matrices(classification_results["runs"])
save_figure(confusion_fig, "classification_confusion_matrices.png")
plt.show()

importance_fig = plot_feature_importance(
    classification_results["feature_importance"],
    f"Feature Importance for {best_run['Model']}"
)
save_figure(importance_fig, "classification_feature_importance.png")
plt.show()

threshold_fig = plot_threshold_sweep(threshold_table, float(recommended_threshold["Threshold"]), sample_count=len(y_train))
save_figure(threshold_fig, "classification_threshold_sweep.png")
plt.show()

classification_results["feature_importance"].to_csv(TABLES_DIR / "classification_feature_importance.csv", index=False)


boundary_pipeline = build_classification_pipeline(
    estimator=clone(best_run["Spec"].estimator),
    feature_names=["Glucose", "BMI"],
    scale=best_run["Spec"].scale,
    use_smote=False,
)
boundary_pipeline.set_params(
    **{
        key: value
        for key, value in best_run["BestParams"].items()
        if key in boundary_pipeline.get_params()
    }
)
boundary_source = clean_df[["Glucose", "BMI"]]
boundary_target = clean_df["Outcome"]
boundary_pipeline.fit(boundary_source, boundary_target)
boundary_fig = plot_decision_boundary(boundary_pipeline, boundary_source, boundary_target)
save_figure(boundary_fig, "classification_decision_boundary_glucose_bmi.png")
plt.show()


**Clinical interpretation prompt:**

When reviewing the results, emphasize recall and false negatives in addition to ROC-AUC. In hospital screening or digital triage settings, a missed high-risk patient may delay downstream testing or treatment. False positives still matter because they increase confirmatory testing burden, but the clinical cost of false negatives is often higher in early-risk detection workflows.


## Regression Task

To extend the project beyond binary prediction, the notebook models `Glucose` as a continuous metabolic indicator using the remaining clinical predictors. This regression task explores whether nonlinear methods can improve over a linear baseline and whether such estimation could support decision support systems when a direct laboratory measurement is delayed or unavailable.


In [9]:
X_reg, y_reg = get_regression_frame(clean_df, target_column="Glucose")
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

regression_results = run_regression_experiment(
    X_reg_train,
    X_reg_test,
    y_reg_train,
    y_reg_test,
)
regression_table = regression_results["comparison_table"]
best_regression_run = regression_results["best_run"]

display(regression_table)
regression_table.to_csv(TABLES_DIR / "regression_model_comparison.csv", index=False)
joblib.dump(best_regression_run["BestEstimator"], MODELS_DIR / "best_regressor.joblib")

regression_fig = plot_regression_diagnostics(
    y_reg_test,
    best_regression_run["Predictions"],
    best_regression_run["Residuals"],
)
save_figure(regression_fig, "regression_diagnostics.png")
plt.show()

display(
    Markdown(
        f"**Best regression model:** {best_regression_run['Model']} with RMSE **{best_regression_run['RMSE']:.2f}** "
        f"and R? **{best_regression_run['R2']:.3f}**. Compare this result with the linear baseline to judge whether nonlinear structure materially improves metabolic estimation."
    )
)


,Model,CV Neg-RMSE,Test MAE,Test MSE,Test RMSE,Test R2
0,Linear Regression,-50.4232,33.6808,2395.2627,48.9414,0.2471
1,Random Forest Regressor,-49.5092,33.9950,2569.7449,50.6927,0.1922
2,Gradient Boosting Regressor,-50.8756,34.4129,2700.1885,51.9633,0.1512


**Best regression model:** Linear Regression with RMSE **48.94** and R? **0.247**. Compare this result with the linear baseline to judge whether nonlinear structure materially improves metabolic estimation.

**Regression interpretation prompt:**

Strong nonlinear gains would suggest that the relationship between glucose and the remaining diagnostic variables is not well represented by a purely linear model. In practice, a regression model is not a replacement for direct laboratory measurement, but it may support risk stratification, resource prioritization, or what-if style clinical analytics.


## Clustering Task

Unsupervised learning is used to identify latent patient subgroups using the standardized predictor variables. K-Means is evaluated across multiple cluster counts using both the elbow method and silhouette score, with agglomerative clustering added as a sensitivity check. PCA is used only for visualization, not for the clustering fit itself.


In [10]:
clustering_results = run_clustering_analysis(clean_df)
clustering_eval = clustering_results["evaluation"]
cluster_profile = clustering_results["profile"]
clustered_frame = clustering_results["clustered_frame"]

display(clustering_eval)
display(cluster_profile)

clustering_eval.to_csv(TABLES_DIR / "clustering_k_selection.csv", index=False)
cluster_profile.to_csv(TABLES_DIR / "cluster_profiles.csv", index=False)

cluster_diag_fig = plot_clustering_diagnostics(clustering_eval)
save_figure(cluster_diag_fig, "clustering_diagnostics.png")
plt.show()

pca_fig = plot_pca_clusters(clustered_frame)
save_figure(pca_fig, "clustering_pca_projection.png")
plt.show()

display(
    Markdown(
        f"The clustering workflow selected **k = {clustering_results['selected_k']}**. Use the profile table to translate the clusters into practical phenotypes such as younger lower-risk groups, older higher-BMI groups, or subgroups with pronounced glucose abnormalities."
    )
)


,Clusters,Inertia,Silhouette
0,2,3214.790449,0.444605
1,3,2668.976363,0.189969
2,4,2427.283654,0.167871
3,5,2261.068708,0.151235
4,6,2114.891711,0.152313


,Cluster,Cholesterol,Glucose,HDLChol,CholHDLRatio,Age,Height,Weight,BMI,SystolicBP,DiastolicBP,Waist,Hip,WaistHipRatio,Count,OutcomePrevalence,ClusterLabel,Narrative
0,0,225.288889,237.333333,42.555556,6.011111,58.977778,67.333333,197.288889,30.506667,145.088889,82.177778,41.244444,45.200000,0.914444,45,0.844444,Older / Higher-Bmi / Higher-Glucose,"Cluster 0 is a relatively older, higher-BMI, h..."
1,1,204.875362,90.382609,51.272464,4.330725,45.182609,65.771014,174.814493,28.549855,136.095652,83.434783,37.428986,42.704348,0.877072,345,0.063768,Older / Higher-Bmi / Higher-Glucose,"Cluster 1 is a relatively older, higher-BMI, h..."


The clustering workflow selected **k = 2**. Use the profile table to translate the clusters into practical phenotypes such as younger lower-risk groups, older higher-BMI groups, or subgroups with pronounced glucose abnormalities.

**Clustering interpretation prompt:**

Cluster labels should be treated as analytical phenotypes rather than fixed biological subtypes. The most useful clinical reading is whether one or more clusters show a combination of elevated glucose, higher BMI, older age, and higher observed diabetes prevalence that could motivate targeted follow-up or education campaigns.


## Explainable AI

Explainability is essential in healthcare because a predictive system should support, not obscure, human judgment. The notebook therefore uses SHAP for global and local explanation of the strongest tree-based classifier when the package is available. If SHAP is not installed, the workflow falls back to permutation importance so that transparency remains available in lighter environments.


In [5]:
xai_results = build_explainability_outputs(
    best_tree_run["BestEstimator"],
    X_train,
    X_test,
    y_test,
)

display(xai_results["local_features"])
display(Markdown(f"**Local explanation note:** {xai_results['case_note']}"))

if xai_results["method"] == "shap":
    import shap

    shap_package = xai_results["shap_package"]

    plt.close("all")
    shap.plots.beeswarm(shap_package["explanation"], max_display=10, show=False)
    plt.title(f"{best_tree_run['Model']} SHAP Values (n={len(X_test)})")
    beeswarm_fig = plt.gcf()
    save_figure(beeswarm_fig, "shap_beeswarm.png")
    plt.show()

    plt.close("all")
    shap.plots.bar(shap_package["explanation"], max_display=10, show=False)
    shap_bar_fig = plt.gcf()
    save_figure(shap_bar_fig, "shap_bar_importance.png")
    plt.show()

    local_position = list(shap_package["explanation_frame"].index).index(xai_results["case_index"])
    plt.close("all")
    shap.plots.waterfall(shap_package["explanation"][local_position], max_display=10, show=False)
    shap_waterfall_fig = plt.gcf()
    save_figure(shap_waterfall_fig, "shap_local_waterfall.png")
    pd.DataFrame({"Feature": shap_package["explanation"].feature_names, "MeanAbsoluteSHAP": np.abs(shap_package["explanation"].values).mean(axis=0)}).sort_values("MeanAbsoluteSHAP", ascending=False).to_csv(TABLES_DIR / "shap_global_importance.csv", index=False)
    pd.DataFrame({"Feature": shap_package["explanation"].feature_names, "Value": shap_package["explanation"].data[local_position], "SHAP": shap_package["explanation"].values[local_position]}).to_csv(TABLES_DIR / "shap_local_contributions.csv", index=False)
    plt.show()
else:
    permutation_importance = xai_results["permutation_importance"]
    permutation_importance.to_csv(TABLES_DIR / "permutation_importance_fallback.csv", index=False)
    permutation_fig = plot_feature_importance(
        permutation_importance,
        f"Permutation Importance for {best_tree_run['Model']}"
    )
    save_figure(permutation_fig, "permutation_importance_fallback.png")
    plt.show()


,Cholesterol,Glucose,HDLChol,CholHDLRatio,Age,Height,Weight,BMI,SystolicBP,DiastolicBP,Waist,Hip,WaistHipRatio,Gender
353,289,111,50,5.8,70,60,220,43.0,126,80,51,54,0.94,Female


**Local explanation note:** False negative case chosen to highlight missed-risk behavior.

## Practical Applications

The workflow supports several realistic healthcare use cases:

- **Hospital screening systems:** prioritize high-risk patients for confirmatory testing when laboratory or clinician time is limited.
- **Mobile health apps:** provide preliminary risk stratification and behavioral intervention prompts.
- **Insurance and care management:** identify members who may benefit from preventive outreach while monitoring fairness and calibration.
- **Public health campaigns:** profile high-risk subgroups for targeted education on weight management, physical activity, and glucose monitoring.
- **Personalized intervention planning:** translate cluster and SHAP outputs into patient-centered lifestyle counseling themes.


## Limitations

This dataset is small, historically overused, and not representative of all populations. It lacks richer covariates such as medications, socioeconomic context, laboratory history, diet, comorbidities, and longitudinal follow-up. The use of a single-source tabular dataset also limits generalizability, and any healthcare deployment would require external validation, fairness assessment, recalibration monitoring, and strong governance around privacy and informed use.


## Future Work

Strong next steps include larger multi-center cohorts, temporal or multimodal modeling, deep learning for richer nonlinear structure, federated learning for privacy-preserving collaboration, richer uncertainty quantification, and more clinically grounded explainability frameworks that combine feature attributions with causal reasoning or counterfactual analysis.


## Key Insights

The table below condenses the most decision-relevant findings from the study after execution.


In [12]:
top_features = ", ".join(classification_results["feature_importance"]["Feature"].head(3).tolist())
cluster_summary = " | ".join(cluster_profile["Narrative"].tolist())
key_insights = pd.DataFrame(
    [
        {
            "Insight": "Best classifier",
            "Finding": f"{best_run['Model']} was selected by training CV ROC-AUC ({best_run['CVBestScore']:.3f}); test ROC-AUC was {best_run['ROCAUC']:.3f}.",
            "Healthcare implication": "Use this model as the primary predictive candidate, then evaluate thresholding for screening safety.",
        },
        {
            "Insight": "Top predictive features",
            "Finding": top_features,
            "Healthcare implication": "These variables deserve close attention in clinical assessment and model monitoring.",
        },
        {
            "Insight": "Threshold recommendation",
            "Finding": f"Training-only OOF selected threshold {threshold_test['Threshold']:.2f}; held-out recall {threshold_test['Recall']:.3f}, precision {threshold_test['Precision']:.3f}, FN {threshold_test['Counts']['fn']}, FP {threshold_test['Counts']['fp']}.",
            "Healthcare implication": "Recall-sensitive deployment may be preferable when missed high-risk patients are costly.",
        },
        {
            "Insight": "Best regression model",
            "Finding": f"{best_regression_run['Model']} achieved RMSE {best_regression_run['RMSE']:.2f} and R? {best_regression_run['R2']:.3f} for glucose prediction.",
            "Healthcare implication": "Nonlinear estimation can support supportive analytics when a direct glucose reading is unavailable.",
        },
        {
            "Insight": "Cluster structure",
            "Finding": cluster_summary,
            "Healthcare implication": "Latent phenotypes can guide targeted follow-up, education, and stratified intervention design.",
        },
    ]
)
display(key_insights)
key_insights.to_csv(TABLES_DIR / "key_insights.csv", index=False)


,Insight,Finding,Healthcare implication
0,Best classifier,Random Forest was selected by training CV ROC-...,Use this model as the primary predictive candi...
1,Top predictive features,"Glucose, Age, SystolicBP",These variables deserve close attention in cli...
2,Threshold recommendation,Training-only OOF selected threshold 0.45; hel...,Recall-sensitive deployment may be preferable ...
3,Best regression model,Linear Regression achieved RMSE 48.94 and R? 0...,Nonlinear estimation can support supportive an...
4,Cluster structure,"Cluster 0 is a relatively older, higher-BMI, h...",Latent phenotypes can guide targeted follow-up...


## Conclusion

The evidence supports exploratory recorded-status assessment. Training CV selects Random Forest, while other candidates' test AUCs remain descriptive comparisons. The fixed threshold misses six of 12 diabetes cases; its uncertainty does not support a screening-ready claim. The Glucose ablation demonstrates substantial reliance on a contemporaneous metabolic marker. External validation is needed; SHAP contributions are descriptive, not causal.


## Appendix: How to Run the Code

1. Place the Kaggle CSV file at `data/raw/diabetes.csv`.
2. Install dependencies from `requirements.txt` using the bundled Python runtime or a local virtual environment.
3. Launch Jupyter and run the notebook from top to bottom.
4. Review generated tables under `outputs/tables/`, figures under `outputs/figures/`, and saved models under `outputs/models/`.

Suggested commands:

```powershell
& "C:/Users/mhral/.cache/codex-runtimes/codex-primary-runtime/dependencies/python/python.exe" -m venv .venv
./.venv/Scripts/python.exe -m pip install -r requirements.txt
./.venv/Scripts/python.exe -m jupyter lab
```


In [ ]:
display(Markdown("## Reviewer revision analyses\nThe revision adds test bootstrap confidence intervals, clinical metrics and a Glucose ablation. Selection is training-only; CV tuning scores are not nested generalization estimates. The primary-label audit finds agreement with glycosylated hemoglobin >7.0, with upstream filling of some missing measurements."))
from src.reviewer_revision import run_reviewer_revision
revision_outputs = run_reviewer_revision()
display(pd.read_csv(TABLES_DIR / "glucose_ablation_comparison.csv"))
display(pd.read_csv(TABLES_DIR / "revision_classification_confidence_intervals.csv"))
display(pd.read_csv(TABLES_DIR / "revision_smote_results.csv"))


## Nested complete classification-pipeline evaluation
Five outer stratified folds evaluate model-family, hyperparameter, and threshold selection using only the existing 312 training patients. Inside each outer-training partition, five-fold inner CV repeated three times tunes all four families. Threshold selection uses another five-fold training cross-fit with hyperparameter retuning. Outer validation is used only after all choices are locked. The original 78-patient test evaluation is preserved. Fold means and sample SD are descriptive, not confidence intervals.


In [ ]:
from run_nested_pipeline import main as run_nested_pipeline
run_nested_pipeline()
display(pd.read_csv(TABLES_DIR / "nested_pipeline_outer_folds.csv"))
display(pd.read_csv(TABLES_DIR / "nested_pipeline_summary.csv"))
display(pd.read_csv(TABLES_DIR / "nested_pipeline_selection_frequency.csv"))
